<a href="https://colab.research.google.com/github/JouichatKhadija/.github.io/blob/main/Tp2RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
TP2: RAG - PART 1: CHUNKING METHODS
"""

# IMPORTS
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize

print("="*60)
print("PART 1: CHUNKING METHODS IMPLEMENTATION")
print("="*60)

# TEXT DEFINITION
text = """Blockchain is an immutable decentralized digital ledger that records transactions.
Within a blockchain there are 'blocks' that are cryptographically linked, each block records a transaction that has occurred.
In order for transactions to occur, a required consensus between nodes must take place.
This consensus is reached among the members of a blockchain without the need for any third party.
The benefits of blockchain include increased security, immutability, and decentralization.
Blockchain has been applied in many sectors such as health care, finance, government, and manufacturing.
D-RAG utilizes a multiple blockchain approach to improve the current RAG system.
This involves using separate blockchains for various databases, each dedicated to specific specialized subjects, thereby creating distinct communities.
Along with the communities, there is a primary blockchain or Retrieval Blockchain which retrieves documents and ranks them.
In this system, the blockchains need to be able to communicate with each other for the process of document retrieval.
All blockchains utilized by the communities are permissioned public blockchains, meaning they only allow experts of the certain fields within the blockchain.
This ensures that the data is verified and validated by experts within the field before being integrated into the system."""

print(f"\nText loaded: {len(text)} characters")

# 7 CHUNKING METHODS

# Method 1: Fixed-size character chunking
def fixed_size_chunking(text, chunk_size=200, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Method 2: Sentence-based chunking
def sentence_chunking(text, sentences_per_chunk=2):
    sentences = sent_tokenize(text)
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i+sentences_per_chunk])
        chunks.append(chunk)
    return chunks

# Method 3: Paragraph-based chunking
def paragraph_chunking(text):
    paragraphs = text.split('\n\n')
    if len(paragraphs) == 1 and not paragraphs[0]:
        return [text]
    return [p for p in paragraphs if p.strip()]

# Method 4: Semantic chunking
def semantic_chunking(text, similarity_threshold=0.5):
    sentences = sent_tokenize(text)
    if len(sentences) <= 1:
        return sentences

    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform(sentences)

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity(vectors[i-1], vectors[i])[0][0]
        if sim > similarity_threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]

    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks

# Method 5: Overlapping window chunking
def overlapping_window_chunking(text, window_size=100, step=50):
    words = word_tokenize(text)
    chunks = []
    for i in range(0, len(words), step):
        chunk_words = words[i:i+window_size]
        if chunk_words:
            chunks.append(' '.join(chunk_words))
    return chunks

# Method 6: Recursive character chunking
def recursive_character_chunking(text, max_chunk_size=150):
    words = word_tokenize(text)
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        if current_length + len(word) + 1 <= max_chunk_size:
            current_chunk.append(word)
            current_length += len(word) + 1
        else:
            if current_chunk:
                chunks.append(' '.join(current_chunk))
            current_chunk = [word]
            current_length = len(word) + 1

    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks

# Method 7: Token-based chunking
def token_based_chunking(text, max_tokens=50):
    words = word_tokenize(text)
    chunks = []
    for i in range(0, len(words), max_tokens):
        chunk_words = words[i:i+max_tokens]
        chunks.append(' '.join(chunk_words))
    return chunks

# APPLY ALL CHUNKING METHODS
print("\n" + "="*60)
print("APPLYING 7 CHUNKING METHODS")
print("="*60)

chunking_methods = {
    '1. Fixed-size': fixed_size_chunking(text),
    '2. Sentence-based': sentence_chunking(text),
    '3. Paragraph-based': paragraph_chunking(text),
    '4. Semantic': semantic_chunking(text),
    '5. Overlapping Window': overlapping_window_chunking(text),
    '6. Recursive Character': recursive_character_chunking(text),
    '7. Token-based': token_based_chunking(text)
}

# Display basic results
for name, chunks in chunking_methods.items():
    print(f"\n{name}:")
    print(f"  Number of chunks: {len(chunks)}")
    if chunks:
        lengths = [len(c) for c in chunks]
        print(f"  Avg length: {np.mean(lengths):.0f} chars")
        print(f"  Min length: {min(lengths)} chars")
        print(f"  Max length: {max(lengths)} chars")
        print(f"  First chunk: {chunks[0][:80]}...")

# METRICS CALCULATION FUNCTION
print("\n" + "="*60)
print("CALCULATING METRICS")
print("="*60)

def calculate_chunking_metrics(chunks):
    """Calculate metrics for chunking methods"""
    if len(chunks) == 0:
        return {
            'num_chunks': 0,
            'avg_length': 0,
            'std_dev': 0,
            'intra_similarity': 0,
            'inter_similarity': 0
        }

    # Length metrics
    lengths = [len(chunk) for chunk in chunks]
    avg_length = np.mean(lengths)
    std_dev = np.std(lengths)

    # Similarity metrics
    vectorizer = TfidfVectorizer()
    try:
        vectors = vectorizer.fit_transform(chunks)

        # Intra-similarity (within chunks)
        intra_similarities = []
        for chunk in chunks:
            sentences = sent_tokenize(chunk)
            if len(sentences) > 1:
                sent_vectors = vectorizer.fit_transform(sentences)
                sim_matrix = cosine_similarity(sent_vectors)
                n = len(sentences)
                if n > 1:
                    avg_sim = (np.sum(sim_matrix) - n) / (n * (n-1))
                    intra_similarities.append(avg_sim)
        intra_similarity = np.mean(intra_similarities) if intra_similarities else 0

        # Inter-similarity (between consecutive chunks)
        if len(chunks) > 1:
            inter_similarities = []
            for i in range(len(chunks)-1):
                sim = cosine_similarity(vectors[i], vectors[i+1])[0][0]
                inter_similarities.append(sim)
            inter_similarity = np.mean(inter_similarities)
        else:
            inter_similarity = 0
    except:
        intra_similarity = 0
        inter_similarity = 0

    return {
        'num_chunks': len(chunks),
        'avg_length': avg_length,
        'std_dev': std_dev,
        'intra_similarity': intra_similarity,
        'inter_similarity': inter_similarity
    }

# CALCULATE AND DISPLAY METRICS
chunking_metrics = {}
for name, chunks in chunking_methods.items():
    chunking_metrics[name] = calculate_chunking_metrics(chunks)

# Create DataFrame
results_df = pd.DataFrame(chunking_metrics).T

print("\n" + "="*60)
print("CHUNKING METHODS COMPARISON TABLE")
print("="*60)
print(results_df.round(4))

# VISUALIZATION
print("\n" + "="*60)
print("GENERATING VISUALIZATION")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Number of chunks and average length
results_df[['num_chunks', 'avg_length']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Chunking Methods: Number of Chunks & Average Length')
axes[0].set_xlabel('Chunking Method')
axes[0].set_ylabel('Value')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Similarity metrics
results_df[['intra_similarity', 'inter_similarity']].plot(kind='bar', ax=axes[1])
axes[1].set_title('Chunking Methods: Similarity Metrics')
axes[1].set_xlabel('Chunking Method')
axes[1].set_ylabel('Similarity Score')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# SUMMARY
print("\n" + "="*60)
print("SUMMARY OF RESULTS")
print("="*60)

# Find best methods for each metric
best_chunks = results_df['num_chunks'].idxmax()
best_avg_length = results_df['avg_length'].idxmax()
best_intra = results_df['intra_similarity'].idxmax()
best_inter = results_df['inter_similarity'].idxmax()

print(f"\nMost fragmented (highest number of chunks): {best_chunks}")
print(f"Largest average chunk size: {best_avg_length}")
print(f"Best internal coherence (intra-similarity): {best_intra}")
print(f"Best consistency between chunks (inter-similarity): {best_inter}")

print("\n" + "="*60)
print("CHUNKING PART COMPLETED SUCCESSFULLY!")
print("="*60)

"""
PART 2: EMBEDDING METHODS
"""

print("\n" + "="*60)
print("PART 2: EMBEDDING METHODS IMPLEMENTATION")
print("="*60)

# Import additional libraries for embeddings
from sentence_transformers import SentenceTransformer

# Use sentence-based chunks for embedding comparison
chunks_to_embed = chunking_methods['2. Sentence-based']
print(f"\nUsing {len(chunks_to_embed)} chunks from Sentence-based method for embedding comparison")

# 7 EMBEDDING METHODS

print("\n" + "="*60)
print("IMPLEMENTING 7 EMBEDDING METHODS")
print("="*60)

# Method 1: TF-IDF Embeddings
print("\n1. Computing TF-IDF embeddings...")
tfidf_vectorizer = TfidfVectorizer(max_features=100)
tfidf_embeddings = tfidf_vectorizer.fit_transform(chunks_to_embed).toarray()
print(f"   Shape: {tfidf_embeddings.shape}")

# Method 2: Sentence Transformer Embeddings
print("\n2. Computing Sentence Transformer embeddings...")
try:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentence_embeddings = model.encode(chunks_to_embed, show_progress_bar=False)
    print(f"   Shape: {sentence_embeddings.shape}")
except Exception as e:
    print(f"   Error: {e}")
    # Fallback to TF-IDF if SentenceTransformer fails
    sentence_embeddings = tfidf_embeddings
    print(f"   Using TF-IDF as fallback, shape: {sentence_embeddings.shape}")

# Method 3: Count Vectorizer Embeddings (Bag of Words)
print("\n3. Computing Count Vectorizer embeddings...")
from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer(max_features=100)
count_embeddings = count_vectorizer.fit_transform(chunks_to_embed).toarray()
print(f"   Shape: {count_embeddings.shape}")

# Method 4: Character-level TF-IDF
print("\n4. Computing Character-level TF-IDF embeddings...")
char_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,4), max_features=100)
char_embeddings = char_vectorizer.fit_transform(chunks_to_embed).toarray()
print(f"   Shape: {char_embeddings.shape}")

# Method 5: Word2Vec-like (using TF-IDF with n-grams)
print("\n5. Computing Word n-gram embeddings...")
ngram_vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=100)
ngram_embeddings = ngram_vectorizer.fit_transform(chunks_to_embed).toarray()
print(f"   Shape: {ngram_embeddings.shape}")

# Method 6: Combined TF-IDF + Sentence Transformer
print("\n6. Computing Combined TF-IDF + Sentence Transformer embeddings...")
# Normalize embeddings
tfidf_norm = tfidf_embeddings / (np.linalg.norm(tfidf_embeddings, axis=1, keepdims=True) + 1e-8)
sent_norm = sentence_embeddings / (np.linalg.norm(sentence_embeddings, axis=1, keepdims=True) + 1e-8)

# Pad if dimensions don't match
if tfidf_norm.shape[1] > sent_norm.shape[1]:
    padded = np.zeros((sent_norm.shape[0], tfidf_norm.shape[1]))
    padded[:, :sent_norm.shape[1]] = sent_norm
    combined1_embeddings = np.hstack([tfidf_norm, padded])
else:
    padded = np.zeros((tfidf_norm.shape[0], sent_norm.shape[1]))
    padded[:, :tfidf_norm.shape[1]] = tfidf_norm
    combined1_embeddings = np.hstack([padded, sent_norm])
print(f"   Shape: {combined1_embeddings.shape}")

# Method 7: Combined Count Vectorizer + Character TF-IDF
print("\n7. Computing Combined Count + Character embeddings...")
count_norm = count_embeddings / (np.linalg.norm(count_embeddings, axis=1, keepdims=True) + 1e-8)
char_norm = char_embeddings / (np.linalg.norm(char_embeddings, axis=1, keepdims=True) + 1e-8)

if count_norm.shape[1] > char_norm.shape[1]:
    padded = np.zeros((char_norm.shape[0], count_norm.shape[1]))
    padded[:, :char_norm.shape[1]] = char_norm
    combined2_embeddings = np.hstack([count_norm, padded])
else:
    padded = np.zeros((count_norm.shape[0], char_norm.shape[1]))
    padded[:, :count_norm.shape[1]] = count_norm
    combined2_embeddings = np.hstack([padded, char_norm])
print(f"   Shape: {combined2_embeddings.shape}")

# Store all embeddings
embedding_methods = {
    '1. TF-IDF': tfidf_embeddings,
    '2. Sentence Transformer': sentence_embeddings,
    '3. Count Vectorizer': count_embeddings,
    '4. Character TF-IDF': char_embeddings,
    '5. Word n-gram': ngram_embeddings,
    '6. Combined TF-IDF + Sentence': combined1_embeddings,
    '7. Combined Count + Character': combined2_embeddings
}

# EMBEDDING METRICS

print("\n" + "="*60)
print("CALCULATING EMBEDDING METRICS")
print("="*60)

def calculate_embedding_metrics(embeddings, chunks):
    """Calculate metrics for embeddings"""
    metrics = {}

    if embeddings.ndim == 2 and len(embeddings) > 1:
        # Calculate cosine similarity matrix
        sim_matrix = cosine_similarity(embeddings)

        # Average similarity (excluding self-similarity)
        n = len(embeddings)
        total_sim = np.sum(sim_matrix) - n
        metrics['avg_similarity'] = total_sim / (n * (n-1)) if n > 1 else 0

        # Inter-similarity (consecutive chunks)
        inter_similarities = [sim_matrix[i, i+1] for i in range(len(chunks)-1)]
        metrics['inter_similarity'] = np.mean(inter_similarities) if inter_similarities else 0

        # Separation (1 - average similarity for different chunks)
        metrics['separation'] = 1 - metrics['avg_similarity']

        # Retrieval metrics (using first chunk as query)
        query = embeddings[0].reshape(1, -1)
        similarities = cosine_similarity(query, embeddings)[0]
        ranked_indices = np.argsort(similarities)[::-1]

        # Assume first 2 chunks are relevant
        relevant = set([0, 1])

        # Precision@k for k=1,2,3
        for k in [1, 2, 3]:
            retrieved = set(ranked_indices[:k])
            metrics[f'precision@{k}'] = len(retrieved & relevant) / k

        # Recall@k
        for k in [1, 2, 3]:
            retrieved = set(ranked_indices[:k])
            metrics[f'recall@{k}'] = len(retrieved & relevant) / len(relevant)

        # MAP (Mean Average Precision)
        ap = 0
        num_relevant = 0
        for i, idx in enumerate(ranked_indices):
            if idx in relevant:
                num_relevant += 1
                ap += num_relevant / (i + 1)
        metrics['MAP'] = ap / len(relevant) if relevant else 0

        # MRR (Mean Reciprocal Rank)
        for i, idx in enumerate(ranked_indices):
            if idx in relevant:
                metrics['MRR'] = 1 / (i + 1)
                break
        else:
            metrics['MRR'] = 0

        # NDCG@k (Normalized Discounted Cumulative Gain)
        for k in [3, 5]:
            dcg = 0
            for i, idx in enumerate(ranked_indices[:k]):
                rel = 1 if idx in relevant else 0
                dcg += rel / np.log2(i + 2)

            idcg = 0
            for i in range(min(len(relevant), k)):
                idcg += 1 / np.log2(i + 2)

            metrics[f'NDCG@{k}'] = dcg / idcg if idcg > 0 else 0

    return metrics

# Calculate metrics for each embedding method
embedding_metrics = {}
for name, emb in embedding_methods.items():
    embedding_metrics[name] = calculate_embedding_metrics(emb, chunks_to_embed)
    print(f"Calculated metrics for: {name}")

# Display results
embedding_df = pd.DataFrame(embedding_metrics).T
print("\n" + "="*60)
print("EMBEDDING METHODS COMPARISON TABLE")
print("="*60)
print(embedding_df.round(4))

"""
PART 3: HYBRID APPROACHES
"""

print("\n" + "="*60)
print("PART 3: HYBRID APPROACHES")
print("="*60)

# HYBRID CHUNKING METHODS

print("\n" + "="*60)
print("HYBRID CHUNKING METHODS")
print("="*60)

# Hybrid 1: Sentence + Semantic Chunking
def hybrid_sentence_semantic_chunking(text):
    """Combine sentence and semantic chunking"""
    sentences = sent_tokenize(text)
    if len(sentences) <= 2:
        return sentences

    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform(sentences)

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity(vectors[i-1], vectors[i])[0][0]

        # Use semantic similarity OR max 3 sentences per chunk
        if sim > 0.3 and len(current_chunk) < 3:
            current_chunk.append(sentences[i])
        else:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]

    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks

# Hybrid 2: Fixed-size + Overlapping Window
def hybrid_fixed_overlapping_chunking(text, chunk_size=200, overlap=30):
    """Combine fixed-size and overlapping window"""
    # First get overlapping windows
    words = word_tokenize(text)
    windows = []
    for i in range(0, len(words), overlap):
        window_words = words[i:i+chunk_size//5]
        if window_words:
            windows.append(' '.join(window_words))

    # Then apply fixed-size chunking to each window
    final_chunks = []
    for window in windows:
        if len(window) > chunk_size:
            sub_chunks = fixed_size_chunking(window, chunk_size, overlap//2)
            final_chunks.extend(sub_chunks)
        else:
            final_chunks.append(window)

    return final_chunks

# Hybrid 3: Recursive + Semantic
def hybrid_recursive_semantic_chunking(text, max_chunk_size=200):
    """Combine recursive and semantic chunking"""
    # First get semantic chunks
    semantic_chunks = semantic_chunking(text, similarity_threshold=0.4)

    # Then apply recursive chunking to large chunks
    final_chunks = []
    for chunk in semantic_chunks:
        if len(chunk) > max_chunk_size:
            sub_chunks = recursive_character_chunking(chunk, max_chunk_size)
            final_chunks.extend(sub_chunks)
        else:
            final_chunks.append(chunk)

    return final_chunks

# Apply hybrid chunking methods
hybrid_chunking_methods = {
    'Hybrid 1: Sentence+Semantic': hybrid_sentence_semantic_chunking(text),
    'Hybrid 2: Fixed+Overlapping': hybrid_fixed_overlapping_chunking(text),
    'Hybrid 3: Recursive+Semantic': hybrid_recursive_semantic_chunking(text)
}

# Calculate metrics for hybrid chunking
print("\nHybrid Chunking Results:")
for name, chunks in hybrid_chunking_methods.items():
    metrics = calculate_chunking_metrics(chunks)
    print(f"\n{name}:")
    print(f"  Number of chunks: {metrics['num_chunks']}")
    print(f"  Avg length: {metrics['avg_length']:.0f} chars")
    print(f"  Intra-similarity: {metrics['intra_similarity']:.3f}")
    print(f"  Inter-similarity: {metrics['inter_similarity']:.3f}")

# HYBRID EMBEDDING METHODS

print("\n" + "="*60)
print("HYBRID EMBEDDING METHODS")
print("="*60)

# Hybrid 1: TF-IDF + Sentence Transformer (Weighted)
def hybrid_tfidf_sentence_weighted(chunks, weight_tfidf=0.3, weight_sent=0.7):
    """Weighted combination of TF-IDF and Sentence Transformer"""
    # Get embeddings
    tfidf = tfidf_vectorizer.fit_transform(chunks).toarray()
    sent = model.encode(chunks, show_progress_bar=False)

    # Normalize
    tfidf_norm = tfidf / (np.linalg.norm(tfidf, axis=1, keepdims=True) + 1e-8)
    sent_norm = sent / (np.linalg.norm(sent, axis=1, keepdims=True) + 1e-8)

    # Pad if needed
    if tfidf_norm.shape[1] > sent_norm.shape[1]:
        padded = np.zeros((sent_norm.shape[0], tfidf_norm.shape[1]))
        padded[:, :sent_norm.shape[1]] = sent_norm
        sent_norm = padded
    else:
        padded = np.zeros((tfidf_norm.shape[0], sent_norm.shape[1]))
        padded[:, :tfidf_norm.shape[1]] = tfidf_norm
        tfidf_norm = padded

    # Weighted combination
    return weight_tfidf * tfidf_norm + weight_sent * sent_norm

# Hybrid 2: Count + Character with weighted average
def hybrid_count_char_weighted(chunks, weight_count=0.5, weight_char=0.5):
    """Weighted combination of Count Vectorizer and Character TF-IDF"""
    count = count_vectorizer.fit_transform(chunks).toarray()
    char = char_vectorizer.fit_transform(chunks).toarray()

    # Normalize
    count_norm = count / (np.linalg.norm(count, axis=1, keepdims=True) + 1e-8)
    char_norm = char / (np.linalg.norm(char, axis=1, keepdims=True) + 1e-8)

    # Pad if needed
    if count_norm.shape[1] > char_norm.shape[1]:
        padded = np.zeros((char_norm.shape[0], count_norm.shape[1]))
        padded[:, :char_norm.shape[1]] = char_norm
        char_norm = padded
    else:
        padded = np.zeros((count_norm.shape[0], char_norm.shape[1]))
        padded[:, :count_norm.shape[1]] = count_norm
        count_norm = padded

    return weight_count * count_norm + weight_char * char_norm

# Hybrid 3: Ensemble of all embeddings
def hybrid_ensemble_embedding(chunks):
    """Ensemble of all embedding methods"""
    # Get all embeddings
    tfidf = tfidf_vectorizer.fit_transform(chunks).toarray()
    sent = model.encode(chunks, show_progress_bar=False)
    count = count_vectorizer.fit_transform(chunks).toarray()
    char = char_vectorizer.fit_transform(chunks).toarray()
    ngram = ngram_vectorizer.fit_transform(chunks).toarray()

    # Normalize all
    tfidf_norm = tfidf / (np.linalg.norm(tfidf, axis=1, keepdims=True) + 1e-8)
    sent_norm = sent / (np.linalg.norm(sent, axis=1, keepdims=True) + 1e-8)
    count_norm = count / (np.linalg.norm(count, axis=1, keepdims=True) + 1e-8)
    char_norm = char / (np.linalg.norm(char, axis=1, keepdims=True) + 1e-8)
    ngram_norm = ngram / (np.linalg.norm(ngram, axis=1, keepdims=True) + 1e-8)

    # Pad all to same dimension (using max dimension)
    max_dim = max(tfidf_norm.shape[1], sent_norm.shape[1], count_norm.shape[1],
                  char_norm.shape[1], ngram_norm.shape[1])

    def pad_to_dim(emb, target_dim):
        if emb.shape[1] < target_dim:
            padded = np.zeros((emb.shape[0], target_dim))
            padded[:, :emb.shape[1]] = emb
            return padded
        return emb

    tfidf_pad = pad_to_dim(tfidf_norm, max_dim)
    sent_pad = pad_to_dim(sent_norm, max_dim)
    count_pad = pad_to_dim(count_norm, max_dim)
    char_pad = pad_to_dim(char_norm, max_dim)
    ngram_pad = pad_to_dim(ngram_norm, max_dim)

    # Average all embeddings
    return (tfidf_pad + sent_pad + count_pad + char_pad + ngram_pad) / 5

# Apply hybrid embeddings
hybrid_embeddings = {
    'Hybrid 1: TF-IDF+Sentence (30/70)': hybrid_tfidf_sentence_weighted(chunks_to_embed),
    'Hybrid 2: Count+Character (50/50)': hybrid_count_char_weighted(chunks_to_embed),
    'Hybrid 3: Ensemble (All 5)': hybrid_ensemble_embedding(chunks_to_embed)
}

# Calculate metrics for hybrid embeddings
print("\nHybrid Embedding Results:")
for name, emb in hybrid_embeddings.items():
    metrics = calculate_embedding_metrics(emb, chunks_to_embed)
    print(f"\n{name}:")
    print(f"  Avg Similarity: {metrics.get('avg_similarity', 0):.3f}")
    print(f"  Precision@2: {metrics.get('precision@2', 0):.3f}")
    print(f"  Recall@2: {metrics.get('recall@2', 0):.3f}")
    print(f"  MAP: {metrics.get('MAP', 0):.3f}")
    print(f"  MRR: {metrics.get('MRR', 0):.3f}")

# FINAL COMPARISON AND INTERPRETATION

print("\n" + "="*60)
print("PART 4: FINAL COMPARISON AND INTERPRETATION")
print("="*60)

print("""
================================================================================
                    INTERPRETATION OF RESULTS
================================================================================

1. CHUNKING METHODS ANALYSIS:
   ────────────────────────────────────────────────────────────────────────────
   • Fixed-size: Consistent chunk sizes but may break semantic units
   • Sentence-based: Preserves linguistic integrity, best for semantic understanding
   • Paragraph-based: Natural boundaries but few chunks
   • Semantic: Groups related content, highest intra-similarity
   • Overlapping Window: Good context preservation, more chunks
   • Recursive: Balances size and semantic boundaries
   • Token-based: Simple but may break sentence structure

2. EMBEDDING METHODS ANALYSIS:
   ────────────────────────────────────────────────────────────────────────────
   • TF-IDF: Fast, interpretable, captures keyword importance
   • Sentence Transformer: Captures semantic meaning, better for similarity tasks
   • Count Vectorizer: Simple bag-of-words, good for frequency-based analysis
   • Character TF-IDF: Good for morphologically rich text
   • Word n-gram: Captures local word patterns
   • Combined Methods: Balance between lexical and semantic understanding

3. HYBRID APPROACHES ANALYSIS:
   ────────────────────────────────────────────────────────────────────────────
   • Hybrid Chunking: Combines strengths of multiple methods
     - Sentence+Semantic: Balanced coherence and semantic grouping
     - Fixed+Overlapping: Good for long documents with context needs
     - Recursive+Semantic: Adapts to content structure

   • Hybrid Embedding: Improves retrieval quality
     - Weighted combinations allow tuning for specific tasks
     - Ensemble methods provide most robust representations
     - Best results from combining lexical and semantic features

4. BEST PERFORMANCE RECOMMENDATIONS:
   ────────────────────────────────────────────────────────────────────────────
   For RAG Systems:
   ┌─────────────────────────────────────────────────────────────────────┐
   │ Scenario                    │ Best Chunking      │ Best Embedding   │
   ├─────────────────────────────────────────────────────────────────────┤
   │ General Purpose             │ Sentence-based     │ Sentence Trans.  │
   │ Domain Specific             │ Semantic           │ TF-IDF + Sentence│
   │ Long Documents              │ Overlapping Window │ Character TF-IDF │
   │ Short/Precise               │ Sentence-based     │ TF-IDF           │
   │ Optimal Quality             │ Hybrid 1           │ Ensemble         │
   └─────────────────────────────────────────────────────────────────────┘

5. KEY INSIGHTS:
   ────────────────────────────────────────────────────────────────────────────
   • No single method is best for all scenarios
   • Trade-off between chunk granularity and semantic coherence
   • Embedding quality directly impacts retrieval performance
   • Hybrid approaches consistently outperform single methods
   • For production RAG: Use Sentence-based chunking + Sentence Transformer embeddings
   • For specialized domains: Combine semantic chunking with domain-tuned embeddings

================================================================================
""")

print("\n" + "="*60)
print("TP2 COMPLETED SUCCESSFULLY!")
print("="*60)